In [1]:
from openmmnapshift.utils import parse_BMRB_entry

## Parse an NMR-STAR file to extract Chemical Shifts and write the input file for openmm-napshift

In [2]:
parse_BMRB_entry(26046, "Data/2ND3")

In [4]:
parse_BMRB_entry(50957, "../../openmm-noe/tutorials/Data/9COJ")

## Applying mutations

In [23]:
import MDAnalysis as MDA
chemical_shifts_data = {}
u = MDA.Universe("../../Martini3-NMR/Data/CREB_proteins/Martini3/1DH3_APO/system/all_atom.pdb")
with open("../../Martini3-NMR/Data/CREB_proteins/AllAtom/1DH3_CRE/NMRData/CS.list", "r") as f:
    for line in f.readlines():
        resid = line.split()[0]
        atom = line.split()[1]
        CS = float(line.split()[2])

        key = (resid, u.select_atoms(f"resid {resid} and chainid A")[0].resname,"A")

        if key not in chemical_shifts_data.keys(): chemical_shifts_data[key] = {}
        chemical_shifts_data[key][atom] = CS

from openmmnapshift.utils import ATOM_TYPES, RESIDUE_TYPES
with open(f'Data/1DH3_APO/CRE_CS_top_indexed.txt', 'w') as f:
    suffix = "_fac"
    f.write(f"{'#NUM' :<8}{'AA' :<8}{''.join([f'{atom:<8}{atom+suffix:<8}' for atom in ATOM_TYPES])}{'CHAIN':<8}\n")
    for i in range(2):
        for (resid,restype,chainid), residue_cs_data in chemical_shifts_data.items():
            f.write(f'{int(resid)-276 + 63*i:<8}{RESIDUE_TYPES[restype]:<8}')
            for atom in ATOM_TYPES:
                if atom in residue_cs_data.keys():
                    f.write(f'{residue_cs_data[atom]:<8.3f}{1.0:<8.2f}') 
                else:
                    f.write(f"{'-':<8}{1.0:<8.2f}") 
            f.write(f'{1:<8}\n')

/home/mina/anaconda3/envs/OpenMMNapShift/lib/python3.11/site-packages/MDAnalysis/topology/PDBParser.py:331: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using MDAnalysis.topology.guessers.
  warnings.warn("Element information is missing, elements attribute "


In [20]:
from openmmnapshift.utils import read_chemical_shifts
import numpy as np
CS = read_chemical_shifts(f'Data/1DH3_APO/CRE_CS.txt')

with open("../../Martini3-NMR/Data/CREB_proteins/Martini3/1DH3_APO/NMRData/CRE_formated_CS.list", "w") as f:
    for (resid,chainid), (restype, chemical_shifts, chemical_shifts_factors) in CS.items():
        for atom in ATOM_TYPES:
            cs_val = chemical_shifts[atom]
            if not np.isnan(cs_val):
                f.write(f"{resid:<5}{atom:<3}{chemical_shifts[atom]:<7.3f}\n")

In [13]:
from openmmnapshift.utils import read_chemical_shifts, ATOM_TYPES, RESIDUE_TYPES
import numpy as np
CS = read_chemical_shifts(f'Data/8K6Z/CS.txt')
RC_CS = read_chemical_shifts(f'Data/8K6Z/RC_CS.txt')

combined_CS = {}
for (resid, chainid),(restype, CS_vals, CS_scale) in CS.items():
    RC_CS_vals = RC_CS[k][1]
    combined_CS_vals = {}
    for a in ATOM_TYPES:
        combined_CS_vals[a] = (CS_vals[a] + RC_CS_vals[a])/2

    combined_CS[(resid, restype,chainid)] = combined_CS_vals

with open(f'Data/8K6Z/combined_CS.txt', 'w') as f:
    suffix = "_fac"
    f.write(f"{'#NUM' :<8}{'AA' :<8}{''.join([f'{atom:<8}{atom+suffix:<8}' for atom in ATOM_TYPES])}{'CHAIN':<8}\n")
    for (resid,restype,chainid), residue_cs_data in combined_CS.items():
        f.write(f'{resid:<8}{restype:<8}')
        for atom in ATOM_TYPES:
            if atom in residue_cs_data.keys():
                f.write(f'{residue_cs_data[atom]:<8.3f}{1.0:<8.2f}') 
            else:
                f.write(f"{'-':<8}{1.0:<8.2f}") 
        f.write(f'{chainid:<8}\n')